# 01 — Train Your Custom Qwen3-4B AI (100% Free on Google Colab T4 GPU)

**Instructions (Free Google Colab — T4 GPU):**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run cells 1 to 5 in order.
3. Optional: Upload `training_feedback.jsonl` (exported from your Middle Dashboard) to improve your model on your own corrections!

In [ ]:
# Cell 1: Install Unsloth & dependencies
!pip install -q unsloth trl peft accelerate bitsandbytes datasets

In [ ]:
# Cell 2: Mount Google Drive to save your trained AI adapter
from google.colab import drive
drive.mount('/content/drive')

import os
VERSION_TAG = "v1.1-qlora"  # Change to v1.2-qlora on next round
SAVE_DIR = f"/content/drive/MyDrive/my-custom-ai/adapters/{VERSION_TAG}"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Adapter will be saved to:", SAVE_DIR)

In [ ]:
# Cell 3: Load Qwen3-4B in 4-bit (runs smoothly on Free Colab T4 15GB GPU)
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Cell 4: Prepare Free Reasoning/Instruction Dataset + Optional Your Corrections
from datasets import load_dataset, concatenate_datasets

# 1) Free high-quality instruction dataset (1000 samples for fast learning run)
base_ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:1000]")

# 2) If you exported training_feedback.jsonl from Middle Dashboard, load & merge it!
feedback_path = "/content/training_feedback.jsonl"
if os.path.exists(feedback_path) and os.path.getsize(feedback_path) > 0:
    fb_ds = load_dataset("json", data_files=feedback_path, split="train")
    fb_ds = concatenate_datasets([fb_ds] * 5)
    dataset = concatenate_datasets([base_ds, fb_ds]).shuffle(seed=42)
    print(f"Merged {len(fb_ds)} correction examples with base dataset!")
else:
    dataset = base_ds
    print("No training_feedback.jsonl found; using base instruction dataset.")

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

In [ ]:
# Cell 5: Train & Save Adapter + Training Log
import json
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

run_info = {
    "version": VERSION_TAG,
    "base_model": MODEL_NAME,
    "train_loss": round(stats.training_loss, 4),
    "dataset_size": len(dataset),
}
with open(os.path.join(SAVE_DIR, "training_run.json"), "w") as f:
    json.dump(run_info, f, indent=2)

print("Training complete!", run_info)